# **MODULE 2 — EKSPERIMEN PREPROCESSING**

Notebook ini melengkapi notebook utama (`MODULE_2_MACHINE_LEARNING_TEXT_CLASSIFICATION.ipynb`).

Di sini kita menguji **beberapa alternatif teknik preprocessing** untuk klasifikasi tweet. Kunci percobaannya: **model, arsitektur, split, dan hyperparameter dibuat identik** — hanya tahap preprocessing yang diubah. Dengan begitu perbedaan hasil benar-benar berasal dari preprocessing.

**Metrik yang dilaporkan:** akurasi, macro-F1, dan F1 kelas minoritas (kelas 1).
Dataset ini sangat tidak seimbang (~93% : 7%), jadi **macro-F1 / F1 kelas 1 lebih jujur** daripada akurasi — menebak kelas mayoritas terus saja sudah memberi akurasi ~93%.

Setiap varian dijalankan dengan **2 seed (42 & 7)**, lalu dilaporkan rata-ratanya.

## 1. Setup, import, dan pemuatan data

In [1]:
import os, re, html, time
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.tokenize import word_tokenize
import emoji
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

for _pkg in ['stopwords', 'wordnet', 'omw-1.4', 'punkt', 'punkt_tab']:
    nltk.download(_pkg, quiet=True)

SEEDS = (42, 7)                 # 2 seed agar hasil tidak bergantung keberuntungan
MAX_WORDS, MAX_LEN, EMB = 30000, 20, 64
print('TensorFlow:', tf.__version__)


def find_file(rel_path):
    p = rel_path
    for _ in range(4):
        if os.path.exists(p):
            return p
        p = os.path.join('..', p)
    raise FileNotFoundError(f'{rel_path} tidak ditemukan')


TRAIN_PATH = find_file(os.path.join('dataset', 'assignment', 'train.csv'))
df_raw = pd.read_csv(TRAIN_PATH)
print('Dataset:', df_raw.shape, '| distribusi label:', dict(df_raw.label.value_counts()))

I0000 00:00:1792345830.435885  210661 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792345830.440102  210661 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1792345830.669407  210661 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1792345832.795767  210661 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792345832.798193  210661 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


TensorFlow: 2.21.0
Dataset: (31962, 3) | distribusi label: {0: np.int64(29720), 1: np.int64(2242)}


## 2. Fungsi preprocessing (parameterized)

Semua varian memakai fungsi yang sama, hanya argumennya yang berbeda:

| Argumen | Efek |
|---|---|
| `keep_hashtag_word` | `True` = pertahankan kata hashtag (`#tag` -> `tag`); `False` = buang seluruh hashtag |
| `use_stopwords` | stopword removal (NLTK) |
| `use_lemma` / `use_stem` | lemmatization (WordNet) atau stemming (Porter) |
| `use_slang` | normalisasi slang + ekspansi kontraksi |
| `emoji_txt` | perbaiki encoding (mojibake) + ubah emoji menjadi teks |
| `dedup` | hapus tweet duplikat sebelum split |

In [2]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

CONTRACTIONS = [(r"won't", 'will not'), (r"can't", 'can not'), (r"n't", ' not'),
                (r"'re", ' are'), (r"'ve", ' have'), (r"'ll", ' will'),
                (r"'d", ' would'), (r"'m", ' am'), (r"'s", ' is')]

SLANG_DICT = {
    'u': 'you', 'ur': 'your', 'r': 'are', 'im': 'i', 'dont': 'not', 'cant': 'not',
    'wont': 'not', 'doesnt': 'not', 'didnt': 'not', 'isnt': 'not', 'wasnt': 'not',
    'ive': 'have', 'gr8': 'great', 'bihday': 'birthday', 'pls': 'please',
    'plz': 'please', 'ppl': 'people', 'luv': 'love', 'bc': 'because', 'cuz': 'because',
    'wanna': 'want', 'gonna': 'going', 'gotta': 'got', 'thx': 'thanks',
    'tbh': 'honest', 'idk': 'know', '2': 'to', '4': 'for', 'b4': 'before',
    'nite': 'night', 'tho': 'though',
}


def repair_mojibake(text):
    """Emoji di dataset ini rusak (UTF-8 yang salah dibaca sebagai Latin-1)."""
    try:
        return text.encode('latin-1').decode('utf-8')
    except (UnicodeEncodeError, UnicodeDecodeError):
        return text


def make_text(raw, keep_hashtag_word=True, use_stopwords=True, use_lemma=True,
              use_stem=False, use_slang=False, emoji_txt=False):
    s = str(raw)
    if emoji_txt:
        s = repair_mojibake(s)                       # perbaiki emoji yang rusak
    s = html.unescape(s).lower()                     # lowercase
    s = re.sub(r'http\S+|www\.\S+', ' ', s)          # hapus URL
    s = re.sub(r'@\w+', ' ', s)                      # hapus mention
    if keep_hashtag_word:
        s = re.sub(r'#(\w+)', r'\1', s)              # pertahankan kata hashtag
    else:
        s = re.sub(r'#\w+', ' ', s)                  # buang seluruh hashtag
    if use_slang:
        for pat, rep in CONTRACTIONS:                # ekspansi kontraksi
            s = re.sub(pat, rep, s)
    if emoji_txt:
        s = emoji.demojize(s, delimiters=(' ', ' ')).replace('_', ' ')   # emoji -> teks
    s = re.sub(r'[^a-z\s]', ' ', s)                  # hapus angka, tanda baca, sisa emoji
    s = re.sub(r'\s+', ' ', s).strip()

    tokens = word_tokenize(s)                        # tokenization
    if use_slang:
        tokens = [SLANG_DICT.get(t, t) for t in tokens]
    if use_stopwords:
        tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    else:
        tokens = [t for t in tokens if len(t) > 1]
    if use_stem:
        tokens = [stemmer.stem(t) for t in tokens]
    elif use_lemma:
        tokens = [lemmatizer.lemmatize(t, pos='v') for t in tokens]
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)


print('Contoh (A3):', make_text(df_raw.tweet.iloc[3], keep_hashtag_word=True)[:80])
print('Contoh (V2):', make_text(df_raw.tweet.iloc[3], keep_hashtag_word=True, emoji_txt=True)[:80])

Contoh (A3): model love take time ur
Contoh (V2): model love take time ur mobile phone kiss face smile eye smile face sunglass mou


## 3. Harness: model identik + pelatihan + evaluasi

In [3]:
def build_model():
    """Arsitektur tetap sama untuk SEMUA varian (Embedding + BiGRU)."""
    vec = layers.TextVectorization(max_tokens=MAX_WORDS, output_mode='int',
                                   output_sequence_length=MAX_LEN)
    inp = layers.Input(shape=(1,), dtype=tf.string)
    x = layers.Embedding(MAX_WORDS, EMB, mask_zero=True)(vec(inp))
    x = layers.SpatialDropout1D(0.3)(x)
    x = layers.Bidirectional(layers.GRU(64, activation='tanh'))(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    m = models.Model(inp, out, name='rnn_text_classifier')
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy',
              metrics=['accuracy'])
    return m, vec


_RESULTS = []


def run_variant(name, **cfg):
    dedup = cfg.pop('dedup', False)
    d = df_raw.drop_duplicates(subset='tweet').reset_index(drop=True) if dedup else df_raw
    t0 = time.time()
    texts = [make_text(t, **cfg) for t in d['tweet']]
    X = np.asarray(texts, dtype=object)
    y = np.asarray(d['label'], dtype=int)

    accs, macros, f1mins, vocab = [], [], [], None
    for seed in SEEDS:
        tf.keras.utils.set_random_seed(seed)
        Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.2,
                                              random_state=42, stratify=y)
        model, vec = build_model()
        vec.adapt(Xtr)
        vocab = len(vec.get_vocabulary())
        model.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=15, batch_size=128,
                  callbacks=[callbacks.EarlyStopping(monitor='val_loss', patience=3,
                                                     restore_best_weights=True),
                             callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                                         patience=2, min_lr=1e-5)],
                  verbose=0)
        pred = (model.predict(Xva, verbose=0).ravel() >= 0.5).astype(int)
        accs.append(accuracy_score(yva, pred))
        macros.append(f1_score(yva, pred, average='macro'))
        f1mins.append(f1_score(yva, pred, pos_label=1))

    _RESULTS.append({'Varian': name, 'n': len(d), 'Akurasi (mean)': np.mean(accs),
                     'std': np.std(accs), 'macro-F1': np.mean(macros),
                     'F1 kelas 1': np.mean(f1mins), 'Vocab': vocab,
                     'Waktu (s)': round(time.time() - t0)})
    print(f'selesai: {name}  ({round(time.time() - t0)}s)')

## 4. Eksperimen 1 — Variasi preprocessing dasar

- **A0 baseline** — buang seluruh hashtag + stopword removal + lemmatization
- **A1** — tanpa stopword removal (kata fungsi/negasi dipertahankan)
- **A2** — stemming Porter sebagai ganti lemmatization
- **A3** — pertahankan *kata* hashtag (buang hanya simbol `#`)

In [4]:
_RESULTS = []
run_variant('A0 baseline (stopwords + lemma)',
            keep_hashtag_word=False, use_stopwords=True, use_lemma=True, use_stem=False)
run_variant('A1 tanpa stopword removal',
            keep_hashtag_word=False, use_stopwords=False, use_lemma=True, use_stem=False)
run_variant('A2 stemming (Porter)',
            keep_hashtag_word=False, use_stopwords=True, use_lemma=False, use_stem=True)
run_variant('A3 pertahankan kata hashtag',
            keep_hashtag_word=True, use_stopwords=True, use_lemma=True, use_stem=False)

res1 = pd.DataFrame(_RESULTS)
display(res1.round(4))

E0000 00:00:1792345838.547369  210661 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


E0000 00:00:1792345841.216987  210661 util.cc:131] oneDNN supports DT_BOOL only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


selesai: A0 baseline (stopwords + lemma)  (58s)


selesai: A1 tanpa stopword removal  (69s)


selesai: A2 stemming (Porter)  (55s)


selesai: A3 pertahankan kata hashtag  (67s)


,Varian,n,Akurasi (mean),std,macro-F1,F1 kelas 1,Vocab,Waktu (s)
0,A0 baseline (stopwords + lemma),31962,0.9474,0.0001,0.7524,0.5328,15141,58
1,A1 tanpa stopword removal,31962,0.9500,0.0016,0.7820,0.5907,15249,69
2,A2 stemming (Porter),31962,0.9461,0.0005,0.7481,0.5248,14060,55
3,A3 pertahankan kata hashtag,31962,0.9550,0.0021,0.8183,0.6608,28787,67


**Catatan hasil Eksperimen 1:**

- **A3 (pertahankan kata hashtag) menang jelas**, terutama di macro-F1 dan F1 kelas 1. Penyebabnya: hashtag adalah token paling diskriminatif di dataset ini — label 1 banyak berisi `#trump`, `#politics`, `#libtard`, `#sjw`, `#blm`, `#brexit`, `#hate`, sedangkan label 0 banyak berisi `#love`, `#positive`, `#thankful`, `#smile`. Membuang seluruh hashtag = membuang sinyal terkuat. Nama "A3" inilah yang jadi **basis** untuk Eksperimen 2.
- **A1 (tanpa stopword removal)** sedikit lebih baik dari baseline: tweet di sini pendek (median 12 kata), dan stopword removal membuang ~33% teks sehingga kata fungsi & negasi ("not", "no") yang penting ikut hilang.
- **A2 (stemming) justru turun**: Porter memotong kata tanpa dasar linguistik (`dysfunction` -> `dysfunct`) sehingga over-merging dan makna kabur. Lemmatization lebih tepat.

## 5. Eksperimen 2 — 3 teknik tambahan di atas A3

- **V1** — normalisasi slang + ekspansi kontraksi
- **V2** — perbaikan encoding (mojibake) + emoji -> teks
- **V3** — deduplikasi (hapus tweet kembar sebelum split)
- **V4** — gabungan ketiganya

In [5]:
_RESULTS = []
run_variant('V1 A3 + slang/kontraksi',
            keep_hashtag_word=True, use_stopwords=True, use_lemma=True, use_slang=True)
run_variant('V2 A3 + encoding/emoji',
            keep_hashtag_word=True, use_stopwords=True, use_lemma=True, emoji_txt=True)
run_variant('V3 A3 + deduplikasi',
            keep_hashtag_word=True, use_stopwords=True, use_lemma=True, dedup=True)
run_variant('V4 A3 + semua teknik',
            keep_hashtag_word=True, use_stopwords=True, use_lemma=True,
            use_slang=True, emoji_txt=True, dedup=True)

res2 = pd.DataFrame(_RESULTS)
display(res2.round(4))

selesai: V1 A3 + slang/kontraksi  (67s)


selesai: V2 A3 + encoding/emoji  (69s)


selesai: V3 A3 + deduplikasi  (76s)


selesai: V4 A3 + semua teknik  (79s)


,Varian,n,Akurasi (mean),std,macro-F1,F1 kelas 1,Vocab,Waktu (s)
0,V1 A3 + slang/kontraksi,31962,0.9569,0.0015,0.8274,0.6780,28802,67
1,V2 A3 + encoding/emoji,31962,0.9587,0.0006,0.8275,0.6771,28941,69
2,V3 A3 + deduplikasi,29530,0.9609,0.0041,0.8303,0.6815,28854,76
3,V4 A3 + semua teknik,29530,0.9616,0.0029,0.8335,0.6876,29027,79


**Catatan hasil Eksperimen 2:**

- **Deduplikasi memberi lompatan tunggal terbesar.** Ada 2.432 tweet kembar, dan labelnya konsisten (tidak ada tweet sama dengan label berbeda), jadi aman dihapus. Ini soal **data leakage**: bila tweet identik ada di train *dan* validation, model seolah sudah hafal jawabannya. Setelah duplikat dihapus, angka jadi lebih jujur sekaligus lebih tinggi.
- **Perbaikan encoding + emoji->teks** juga menaikkan hasil. ~10.756 tweet emojinya rusak (UTF-8 salah dibaca sebagai Latin-1, tampil seperti `ð`). Setelah encoding diperbaiki, emoji diubah jadi kata (`😭` -> "loudly crying face") sehingga sinyal emosi yang tadinya terbuang jadi terpakai.
- **Normalisasi slang/kontraksi** menaikkan hasil lebih kecil tapi konsisten: menyatukan bentuk yang sebenarnya satu kata (`bihday`->`birthday`, `u`->`you`, `gr8`->`great`).
- **V4 (gabungan ketiganya) paling baik** — efeknya saling menumpuk.

**Kesimpulan:** urutan terbaik adalah **V4 > V3 > V2 > V1 > A3**. Konfigurasi V4 inilah yang dipakai di notebook utama.